# ClashBot universal control notebook

All cells are explicit. Recognition and planning cells are read-only; action cells are marked clearly.

In [ ]:
from pathlib import Path
import subprocess, sys, json
ROOT = Path.cwd()
if not (ROOT / 'src' / 'clashbot').is_dir(): ROOT = Path.cwd().parent
PYTHON = ROOT / '.venv' / 'Scripts' / 'python.exe'
SERIAL = '127.0.0.1:21513'  # change for another MEmu instance
def bot(*args, check=True):
    p = subprocess.run([str(PYTHON), '-m', 'clashbot', *map(str,args)], cwd=ROOT, text=True, capture_output=True)
    print(p.stdout, end=''); print(p.stderr, end='', file=sys.stderr)
    if check and p.returncode: raise RuntimeError(p.stderr or p.stdout)
    return p

In [ ]:
# Read-only: confirm the emulator and save a baseline screenshot
bot('devices', check=False)
capture = ROOT / 'captures' / 'notebook' / 'current.png'
capture.parent.mkdir(parents=True, exist_ok=True)
bot('screenshot', SERIAL, capture)

In [ ]:
# Refresh the local level-sorted retrieval index and inspect coverage
bot('asset-train')
bot('asset-status', '--label', 'cannon')
bot('asset-status', '--label', 'archer')

In [ ]:
# Read-only autonomous map: zoom/pan/recover and write report.json
report = bot('scan-base', SERIAL, 'notebook_map', '--route', 'right,left,up,down', '--root', ROOT / 'captures' / 'notebook' / 'autonomous', check=False)
print((ROOT / 'captures' / 'notebook' / 'autonomous' / 'notebook_map' / 'report.json').read_text())

In [ ]:
# Read-only base-management decision (no taps)
bot('manage-status', SERIAL, check=False)
bot('upgrade', SERIAL, '--dry-run', '--scans', '1', check=False)

In [ ]:
# Read-only attack preparation: verify controls and troop cards
bot('open-attack', SERIAL, check=False)
bot('check-battle', SERIAL, check=False)
# Do not enable find-match/loot-attack until the target policy is reviewed.

In [ ]:
# Optional maintenance cells (safe dry-runs)
bot('anti-afk', SERIAL, '--loops', '1', '--dry-run', check=False)
bot('collect', SERIAL, '--loops', '1', '--dry-run', check=False)
# Camera controls, when a live view needs recovery:
# bot('zoom-in', SERIAL, '--steps', '1')
# bot('zoom-out', SERIAL, '--steps', '1')

## Safety boundary

The scanner and planner do not infer missing buildings, loot, reachability, or defense power. Unresolved categories in `report.json` require better live templates or labelled gameplay frames; they are not silently accepted.